# Improved U-Net — BUSI Breast Lesion Segmentation

This notebook runs the **standardized leakage-free improved U-Net experiment** under the shared BUSI notebook contract.

**Experiment-specific choices**
- 192×192 grayscale input
- Wider improved U-Net architecture
- BCE + Dice loss
- Dropout 0.10
- Training-only augmentation

**Shared comparison contract**
- Benign and malignant BUSI cases only; normal cases excluded
- Public Google Drive BUSI ZIP mirror
- Deterministic case ordering and 80/20 split with seed 42
- Identical validation membership across baseline and improved runs
- Best-checkpoint evaluation at threshold 0.5
- Canonical mean per-sample foreground Dice and IoU


## 1. Runtime and Reproducibility

Initialize the runtime, set all available random seeds, enable deterministic TensorFlow operations when supported, and report the active environment.


In [ ]:
import os
import random
import sys
from pathlib import Path

import numpy as np
import tensorflow as tf

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    deterministic_status = "enabled"
except Exception as exc:
    deterministic_status = f"not enabled ({exc})"

gpus = tf.config.list_physical_devices("GPU")

print(f"Python: {sys.version.split()[0]}")
print(f"TensorFlow: {tf.__version__}")
print(f"GPU devices: {gpus}")
print(f"Deterministic TensorFlow ops: {deterministic_status}")
print(f"Seed: {SEED}")


## 2. Dataset Download and Extraction

Download the project-hosted **BUSI.zip** mirror from its public Google Drive share link, validate the archive, extract it inside the Colab runtime, and automatically locate the BUSI dataset root.

The Google Drive file is a reproducibility mirror only. The repository README should cite the original BUSI dataset source separately.


In [ ]:
import shutil
import subprocess
import zipfile

try:
    import gdown
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "gdown"],
        check=True,
    )
    import gdown

BUSI_DRIVE_FILE_ID = "1Pl22yxAccHBUfcBCAgMIK1XesDES-Ytl"
BUSI_DRIVE_SHARE_URL = "https://drive.google.com/file/d/1Pl22yxAccHBUfcBCAgMIK1XesDES-Ytl/view?usp=sharing"

BUSI_ZIP_PATH = Path("/content/BUSI.zip")
BUSI_EXTRACT_DIR = Path("/content/BUSI_dataset")

# Start from a clean local dataset state on every Run All.
if BUSI_ZIP_PATH.exists():
    BUSI_ZIP_PATH.unlink()
if BUSI_EXTRACT_DIR.exists():
    shutil.rmtree(BUSI_EXTRACT_DIR)

print("Downloading BUSI.zip from the public Google Drive mirror...")
downloaded_path = gdown.download(
    id=BUSI_DRIVE_FILE_ID,
    output=str(BUSI_ZIP_PATH),
    quiet=False,
)

if downloaded_path is None or not BUSI_ZIP_PATH.exists():
    raise RuntimeError("BUSI.zip download failed.")

if not zipfile.is_zipfile(BUSI_ZIP_PATH):
    raise RuntimeError(
        "Downloaded file is not a valid ZIP archive. "
        "Check that the Google Drive file is shared for public link access."
    )

BUSI_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUSI_ZIP_PATH, "r") as archive:
    archive.extractall(BUSI_EXTRACT_DIR)

required_class_dirs = ("benign", "malignant", "normal")
candidate_dirs = [BUSI_EXTRACT_DIR] + sorted(
    [path for path in BUSI_EXTRACT_DIR.rglob("*") if path.is_dir()],
    key=lambda path: len(path.parts),
)

DATASET_ROOT = next(
    (
        path
        for path in candidate_dirs
        if all((path / class_name).is_dir() for class_name in required_class_dirs)
    ),
    None,
)

if DATASET_ROOT is None:
    raise FileNotFoundError(
        "Could not locate a BUSI dataset root containing "
        "benign/, malignant/, and normal/ after extraction."
    )

print(f"Downloaded ZIP: {BUSI_ZIP_PATH} ({BUSI_ZIP_PATH.stat().st_size / 1024**2:.2f} MB)")
print(f"Extracted dataset root: {DATASET_ROOT}")


## 3. Dataset Validation

Validate the expected BUSI case counts before preprocessing. Original ultrasound images are counted separately from mask files.


In [ ]:
CLASS_NAMES = ("benign", "malignant", "normal")


def count_original_images(class_dir: Path) -> int:
    return sum(
        1
        for path in class_dir.glob("*.png")
        if "_mask" not in path.stem.lower()
    )


class_counts = {
    class_name: count_original_images(DATASET_ROOT / class_name)
    for class_name in CLASS_NAMES
}

total_cases = sum(class_counts.values())
experiment_cases = class_counts["benign"] + class_counts["malignant"]

print("BUSI case counts:")
for class_name, count in class_counts.items():
    print(f"  {class_name:9s}: {count}")
print(f"  {'total':9s}: {total_cases}")
print(f"Experiment cases (benign + malignant): {experiment_cases}")

assert class_counts == {
    "benign": 437,
    "malignant": 210,
    "normal": 133,
}, f"Unexpected BUSI class counts: {class_counts}"
assert total_cases == 780
assert experiment_cases == 647


## 4. Project Source and Methodology Checks

Clone the exact project source used by this experiment and verify the experiment-specific methodology required by the shared baseline/improved comparison contract.


In [ ]:
import re
import shutil
import subprocess

REPO_URL = "https://github.com/armin-datasci/breast-ultrasound-segmentation-unet.git"
REPO_BRANCH = "fix/leakage-free-improved-unet"
SOURCE_COMMIT = "0cd7317"

REPO_ROOT = Path("/content/breast-ultrasound-segmentation-unet")
if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

subprocess.run(
    ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)],
    check=True,
)
subprocess.run(
    ["git", "checkout", SOURCE_COMMIT],
    cwd=REPO_ROOT,
    check=True,
)

source_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_ROOT,
    text=True,
).strip()

PROJECT_DIR = REPO_ROOT / "02-improved-busi-unet"
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.preprocessing import preprocess_dataset
from src.augmentation import augment_sample
from src.model import unet_nn
from src.train_pipeline import train_pipeline
from src.utils import plot_training_loss, plot_validation_metrics, save_figure
from src.evaluation import mean_dice, mean_iou

preprocessing_source = (PROJECT_DIR / "src" / "preprocessing.py").read_text(encoding="utf-8")
evaluation_source = (PROJECT_DIR / "src" / "evaluation.py").read_text(encoding="utf-8")
training_source = (PROJECT_DIR / "src" / "train_pipeline.py").read_text(encoding="utf-8")
training_compact = re.sub(r"\s+", "", training_source)

forbidden_roi_terms = ("extract_roi", "img_roi", "mask_roi")
methodology_checks = {
    "exact source commit": source_commit.startswith(SOURCE_COMMIT),
    "no ground-truth ROI extraction": not any(
        term in preprocessing_source for term in forbidden_roi_terms
    ),
    "deterministic file ordering": "sorted(" in preprocessing_source,
    "image interpolation is linear": "INTER_LINEAR" in preprocessing_source,
    "mask interpolation is nearest-neighbor": "INTER_NEAREST" in preprocessing_source,
    "canonical mean Dice exists": "def mean_dice" in evaluation_source,
    "canonical mean IoU exists": "def mean_iou" in evaluation_source,
    "best-only checkpoint configured": "save_best_only=True" in training_compact,
    "best checkpoint is not overwritten after fit": "model.save(save_path)" not in training_compact,
}

for name, passed in methodology_checks.items():
    print(f"[{'OK' if passed else 'FAIL'}] {name}")
assert all(methodology_checks.values()), "One or more methodology checks failed."

print(f"\nExact source commit: {source_commit}")


## 5. Experiment Configuration

Define the experiment-specific model configuration while preserving the shared dataset, split, training-budget, reproducibility, and evaluation contracts.


In [ ]:
NOTEBOOK_CONTRACT_VERSION = "1.0"
TASK_DESCRIPTION = (
    "Binary breast-lesion segmentation on benign and malignant "
    "BUSI ultrasound cases; normal cases excluded."
)
EXPERIMENT_NAME = "improved-busi-unet-leakage-free"
MODEL_NAME = "Improved U-Net"
LOSS_NAME = "BCE + Dice"

IMG_SIZE = (192, 192)
INPUT_SHAPE = (192, 192, 1)
BATCH_SIZE = 8
MAX_EPOCHS = 50
LEARNING_RATE = 1e-4
DROPOUT_RATE = 0.10
VAL_FRACTION = 0.20
THRESHOLD = 0.5

ARTIFACT_ROOT = Path(f"/content/busi-unet-final/{EXPERIMENT_NAME}")
FIGURES_DIR = ARTIFACT_ROOT / "figures"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = ARTIFACT_ROOT / "best_model.keras"
METADATA_PATH = ARTIFACT_ROOT / "run_metadata.json"
HISTORY_PATH = ARTIFACT_ROOT / "training_history.json"
VALIDATION_SPLIT_PATH = ARTIFACT_ROOT / "validation_split.json"

config = {
    "input_shape": list(INPUT_SHAPE),
    "batch_size": BATCH_SIZE,
    "maximum_epochs": MAX_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "dropout": DROPOUT_RATE,
    "validation_fraction": VAL_FRACTION,
    "threshold": THRESHOLD,
    "seed": SEED,
    "augmentation_scope": "training_only",
}

for key, value in config.items():
    print(f"{key}: {value}")


## 6. Preprocessing

Load and preprocess BUSI using the experiment-specific pipeline. Common postconditions are enforced: 647 experiment cases, `float32` tensors, normalized images, and binary masks.


In [ ]:
X_full, y_full = preprocess_dataset(
    str(DATASET_ROOT),
    img_height=IMG_SIZE[0],
    img_width=IMG_SIZE[1],
)

print(f"Images shape: {X_full.shape}")
print(f"Masks shape:  {y_full.shape}")
print(f"Image dtype: {X_full.dtype}")
print(f"Mask dtype:  {y_full.dtype}")
print(f"Image range: [{X_full.min():.3f}, {X_full.max():.3f}]")
print(f"Mask values: {np.unique(y_full)}")

assert X_full.shape == (647, 192, 192, 1)
assert y_full.shape == (647, 192, 192, 1)
assert X_full.dtype == np.float32
assert y_full.dtype == np.float32
assert X_full.min() >= 0.0 and X_full.max() <= 1.0
assert set(np.unique(y_full).tolist()).issubset({0.0, 1.0})


## 7. Train / Validation Split

Create the shared deterministic 80/20 split from case indices using `random_state=42`. Save the validation case IDs and a SHA-256 membership fingerprint so the baseline and improved runs can prove they use the same validation cases.


In [ ]:
import hashlib
from sklearn.model_selection import train_test_split

# Reconstruct the exact deterministic case order used by both loaders.
case_ids = np.asarray(
    [
        f"{class_name}/{path.name}"
        for class_name in ("benign", "malignant")
        for path in sorted(
            (DATASET_ROOT / class_name).glob("*.png"),
            key=lambda item: item.name,
        )
        if "_mask" not in path.stem.lower()
    ],
    dtype=object,
)

assert case_ids.shape[0] == X_full.shape[0] == 647

dataset_case_order_sha256 = hashlib.sha256(
    "\n".join(case_ids.tolist()).encode("utf-8")
).hexdigest()

all_indices = np.arange(X_full.shape[0])
train_indices, val_indices = train_test_split(
    all_indices,
    test_size=VAL_FRACTION,
    random_state=SEED,
    shuffle=True,
)

X_train = X_full[train_indices]
y_train = y_full[train_indices]
X_val = X_full[val_indices]
y_val = y_full[val_indices]

validation_case_ids = case_ids[val_indices].tolist()
validation_membership_sha256 = hashlib.sha256(
    "\n".join(sorted(validation_case_ids)).encode("utf-8")
).hexdigest()

validation_split_manifest = {
    "seed": SEED,
    "validation_fraction": VAL_FRACTION,
    "training_samples": int(X_train.shape[0]),
    "validation_samples": int(X_val.shape[0]),
    "dataset_case_order_sha256": dataset_case_order_sha256,
    "validation_membership_sha256": validation_membership_sha256,
    "validation_case_ids": validation_case_ids,
}

print(f"Training samples:   {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")
print(f"Dataset case-order SHA-256: {dataset_case_order_sha256}")
print(f"Validation membership SHA-256: {validation_membership_sha256}")

assert X_train.shape[0] == 517
assert X_val.shape[0] == 130


## 8. Training Data Preparation

Prepare the training partition according to the experiment-specific augmentation policy. The validation partition must remain untouched.


In [ ]:
X_augmented = []
y_augmented = []

for image, mask in zip(X_train, y_train):
    X_augmented.append(image)
    y_augmented.append(mask)

    augmented_image, augmented_mask = augment_sample(image, mask)
    X_augmented.append(augmented_image)
    y_augmented.append(augmented_mask)

X_train_final = np.asarray(X_augmented, dtype=np.float32)
y_train_final = np.asarray(y_augmented, dtype=np.float32)

augmentation_metadata = {
    "scope": "training_only",
    "generated_samples": int(X_train_final.shape[0] - X_train.shape[0]),
    "training_samples_after_augmentation": int(X_train_final.shape[0]),
    "transforms": {
        "horizontal_flip_probability": 0.5,
        "vertical_flip_probability": 0.5,
        "rotation_probability": 0.5,
        "rotation_degrees": [-15, 15],
        "intensity_probability": 0.5,
        "intensity_factor": [0.9, 1.1],
    },
}

print("Augmentation scope: training_only")
print(f"Original training samples: {X_train.shape[0]}")
print(f"Generated augmented samples: {augmentation_metadata['generated_samples']}")
print(f"Training samples after preparation: {X_train_final.shape[0]}")
print(f"Validation samples: {X_val.shape[0]} (untouched)")

assert X_train_final.shape[0] == 1034
assert y_train_final.shape[0] == 1034
assert X_val.shape[0] == 130


## 9. Model Construction

Build the experiment model from the configured input shape and dropout rate, then report the parameter count.


In [ ]:
model = unet_nn(
    input_shape=INPUT_SHAPE,
    dropout_rate=DROPOUT_RATE,
)

print(f"Model: {MODEL_NAME}")
print(f"Trainable model parameters: {model.count_params():,}")


## 10. Training

Train for at most 50 epochs. The experiment-specific training pipeline must preserve the best `ModelCheckpoint` selected by monitored validation soft Dice.


In [ ]:
model, history = train_pipeline(
    model=model,
    X_train=X_train_final,
    y_train=y_train_final,
    X_val=X_val,
    y_val=y_val,
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    callbacks=[],
    save_path=str(BEST_MODEL_PATH),
)

if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f"Best checkpoint was not created: {BEST_MODEL_PATH}")

print(f"Epochs completed: {len(history.epoch)}")
print(f"Best checkpoint: {BEST_MODEL_PATH}")


## 11. Best Checkpoint Evaluation

Reload the exact saved best checkpoint and evaluate that model—not the last in-memory epoch.

Canonical Dice and IoU are computed offline as **mean per-sample foreground metrics** after thresholding predictions at `0.5`.


In [ ]:
best_model = tf.keras.models.load_model(
    BEST_MODEL_PATH,
    compile=False,
)

y_val_pred = best_model.predict(
    X_val,
    batch_size=BATCH_SIZE,
    verbose=1,
)

offline_mean_dice = float(
    mean_dice(y_val, y_val_pred, threshold=THRESHOLD)
)
offline_mean_iou = float(
    mean_iou(y_val, y_val_pred, threshold=THRESHOLD)
)

val_soft_dice_history = history.history["val_dice_coefficient"]
best_epoch = int(np.argmax(val_soft_dice_history) + 1)
best_monitored_soft_val_dice = float(np.max(val_soft_dice_history))
epochs_completed = len(val_soft_dice_history)

print(f"Epochs completed: {epochs_completed}")
print(f"Best epoch: {best_epoch}")
print(f"Best monitored soft validation Dice: {best_monitored_soft_val_dice:.6f}")
print(f"Offline mean foreground Dice: {offline_mean_dice:.6f}")
print(f"Offline mean foreground IoU:  {offline_mean_iou:.6f}")


## 12. Results and Artifacts

Save the complete training history, validation split manifest, run metadata, evaluation figures, and best checkpoint, then package the experiment directory as a ZIP archive.


In [ ]:
import json
from datetime import datetime, timezone

loss_fig = plot_training_loss(history, smooth=True)
save_figure(loss_fig, str(FIGURES_DIR), "training_loss.png")

metrics_fig = plot_validation_metrics(
    offline_mean_dice,
    offline_mean_iou,
)
save_figure(metrics_fig, str(FIGURES_DIR), "validation_metrics.png")

serializable_history = {
    key: [float(value) for value in values]
    for key, values in history.history.items()
}

with HISTORY_PATH.open("w", encoding="utf-8") as file:
    json.dump(serializable_history, file, indent=2)

with VALIDATION_SPLIT_PATH.open("w", encoding="utf-8") as file:
    json.dump(validation_split_manifest, file, indent=2)

run_metadata = {
    "notebook_contract_version": NOTEBOOK_CONTRACT_VERSION,
    "experiment": EXPERIMENT_NAME,
    "task": TASK_DESCRIPTION,
    "model": {
        "name": MODEL_NAME,
        "input_shape": list(INPUT_SHAPE),
        "dropout_rate": DROPOUT_RATE,
        "loss": LOSS_NAME,
    },
    "source": {
        "repository": REPO_URL,
        "branch": REPO_BRANCH,
        "commit": source_commit,
    },
    "dataset": {
        "name": "Breast Ultrasound Images Dataset (BUSI)",
        "access_method": "Public Google Drive ZIP download",
        "google_drive_file_id": BUSI_DRIVE_FILE_ID,
        "google_drive_share_url": BUSI_DRIVE_SHARE_URL,
        "total_busi_images": total_cases,
        "experiment_images": experiment_cases,
        "benign_images": class_counts["benign"],
        "malignant_images": class_counts["malignant"],
        "normal_images": class_counts["normal"],
        "dataset_case_order_sha256": dataset_case_order_sha256,
    },
    "split": {
        "seed": SEED,
        "validation_fraction": VAL_FRACTION,
        "training_samples_before_augmentation": int(X_train.shape[0]),
        "validation_samples": int(X_val.shape[0]),
        "validation_membership_sha256": validation_membership_sha256,
    },
    "augmentation": augmentation_metadata,
    "training": {
        "batch_size": BATCH_SIZE,
        "maximum_epochs": MAX_EPOCHS,
        "epochs_completed": epochs_completed,
        "learning_rate": LEARNING_RATE,
        "best_epoch": best_epoch,
        "best_monitored_soft_validation_dice": best_monitored_soft_val_dice,
    },
    "offline_evaluation": {
        "checkpoint": "exact best ModelCheckpoint",
        "threshold": THRESHOLD,
        "metric_scope": "mean per-sample foreground",
        "mean_dice": offline_mean_dice,
        "mean_iou": offline_mean_iou,
    },
    "methodology": {
        "ground_truth_roi_crop": False,
        "validation_augmentation": False,
        "deterministic_file_order": True,
        "best_checkpoint_reloaded": True,
        "shared_validation_split_fingerprinted": True,
    },
    "artifacts": {
        "best_model": str(BEST_MODEL_PATH),
        "training_history": str(HISTORY_PATH),
        "validation_split": str(VALIDATION_SPLIT_PATH),
        "training_loss_figure": str(FIGURES_DIR / "training_loss.png"),
        "validation_metrics_figure": str(FIGURES_DIR / "validation_metrics.png"),
    },
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

with METADATA_PATH.open("w", encoding="utf-8") as file:
    json.dump(run_metadata, file, indent=2)

ARCHIVE_BASE = Path(f"/content/{EXPERIMENT_NAME}")
archive_path = Path(
    shutil.make_archive(
        str(ARCHIVE_BASE),
        "zip",
        root_dir=ARTIFACT_ROOT,
    )
)

print(f"Metadata: {METADATA_PATH}")
print(f"Training history: {HISTORY_PATH}")
print(f"Validation split manifest: {VALIDATION_SPLIT_PATH}")
print(f"Figures: {FIGURES_DIR}")
print(f"Artifact ZIP: {archive_path}")


## 13. Final Run Summary

Report the canonical values and artifact paths used for the final baseline/improved comparison.


In [ ]:
print("=" * 72)
print(f"{MODEL_NAME.upper()} — FINAL STANDARDIZED RUN")
print("=" * 72)
print(f"Notebook contract: {NOTEBOOK_CONTRACT_VERSION}")
print(f"Experiment samples: {experiment_cases}")
print(f"Train / validation: {X_train.shape[0]} / {X_val.shape[0]}")
print(f"Training samples after preparation: {X_train_final.shape[0]}")
print(f"Augmentation scope: {augmentation_metadata['scope']}")
print(f"Validation fingerprint: {validation_membership_sha256}")
print(f"Epochs completed: {epochs_completed}")
print(f"Best epoch: {best_epoch}")
print(f"Best monitored soft validation Dice: {best_monitored_soft_val_dice:.6f}")
print(f"Offline mean foreground Dice: {offline_mean_dice:.6f}")
print(f"Offline mean foreground IoU:  {offline_mean_iou:.6f}")
print(f"Best checkpoint: {BEST_MODEL_PATH}")
print(f"Metadata: {METADATA_PATH}")
print(f"Training history: {HISTORY_PATH}")
print(f"Validation split manifest: {VALIDATION_SPLIT_PATH}")
print(f"Artifact ZIP: {archive_path}")
print("=" * 72)
